# 📦 Pipeline de Obtención, Limpieza y Estandarización del Dataset (10.000 Imágenes)

**Proyecto:** Clasificación Inteligente de Residuos mediante Redes Neuronales Convolucionales (CNN)  
**Objetivo de este cuaderno:**  
1. Descargar de forma automatizada **1.000 fotografías reales por cada una de las 10 nuevas clases** (10.000 imágenes en total).
2. Aplicar un **pipeline riguroso de filtrado computacional** para descartar dibujos, esquemas, infografías, texto excesivo, duplicados y fotos de catálogo.
3. Auditar el dataset resultante con **estadísticas y gráficos de distribución**.
4. Realizar la **estandarización geométrica y de canales de color (RGB)** lista para el entrenamiento del modelo de 19 clases.

## 1. Configuración del Entorno y Dependencias

In [ ]:
# Instalación de librerías esenciales (descomentar si es necesario)
# !pip install --quiet icrawler opencv-python Pillow matplotlib seaborn gradio

import os
import shutil
import logging
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from concurrent.futures import ThreadPoolExecutor
from icrawler.builtin import BingImageCrawler

# Silenciar logs excesivos de red
logging.getLogger("icrawler").setLevel(logging.CRITICAL)
logging.getLogger("urllib3").setLevel(logging.CRITICAL)

print("✅ Entorno configurado correctamente.")

## 2. Definición de Categorías y Criterios de Búsqueda
Se definen las **10 clases suplementarias** con consultas multilingües exhaustivas y exclusiones negativas estrictas (, , , etc.) para asegurar fotos de residuos auténticos.

In [ ]:
CATEGORIAS_10_CLASES = [
    "Pilas_Baterias",
    "RAEE_Electronicos",
    "Medicamentos",
    "Neumaticos_Gomas",
    "Bombillas",
    "Madera_Procesada",
    "Escombros_Ceramica",
    "Aceites_Quimicos",
    "Residuos_Sanitarios",
    "Tetra_Pack"
]

DIR_DATASET = "dataset_suplementario"
DIR_DESCARTES = "dataset_suplementario_descartes"
META_POR_CLASE = 1000

print(f"🎯 Meta: {len(CATEGORIAS_10_CLASES) * META_POR_CLASE:,} imágenes distribuidas en 10 clases suplementarias:")
for idx, cat in enumerate(CATEGORIAS_10_CLASES, 1):
    print(f"   {idx:2d}. {cat}")

## 3. Descarga Masiva Concurrente con Filtrado Nativo
Se utiliza el script modular multihilo , el cual implementa:
- **16 hilos de descarga paralelos** por categoría.
- **Filtro nativo fotográfico** de Bing () que bloquea automáticamente animaciones, vectores y cliparts.

In [ ]:
# Ejecución del pipeline de descarga (puede ejecutarse directamente en segundo plano o desde terminal)
# !python descargar_dataset.py
print("💡 Para ejecutar la descarga completa en terminal: python descargar_dataset.py")

## 4. Algoritmo de Limpieza Computacional y Filtrado Automático
Cada imagen descargada es procesada por una función de control de calidad basada en visión artificial:
1. **Integridad:** Validación de encabezados binarios con Pillow y decodificación con OpenCV.
2. **Resolución mínima:** Mínimo 200x200 px.
3. **Relación de aspecto:** Rango [0.35, 2.85] para descartar banners o tiras publicitarias.
4. **Filtro Anti-Catálogo:** Análisis de bordes para detectar fondos blancos puros de estudio de tienda.
5. **Filtro Anti-Infografía / Dibujo:** Medición de entropía de color y conteo de tonos cuantizados.
6. **Filtro de Texto Excesivo:** Detección morfológica de alta densidad de texto tipo afiche publicitario.
7. **Deduplicación:** Hashing perceptivo () con distancia de Hamming <= 2.

In [ ]:
def calcular_dhash(img_bgr, hash_size=8):
    gris = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    peq = cv2.resize(gris, (hash_size + 1, hash_size))
    diff = peq[:, 1:] > peq[:, :-1]
    return sum([2 ** i for (i, v) in enumerate(diff.flatten()) if v])

def distancia_hamming(h1, h2):
    return bin(h1 ^ h2).count("1")

def evaluar_calidad_imagen(ruta_archivo):
    try:
        with Image.open(ruta_archivo) as pil_img:
            pil_img.verify()
    except Exception:
        return False, "corrupta_pil"

    img_bgr = cv2.imread(ruta_archivo)
    if img_bgr is None:
        return False, "corrupta_opencv"

    h, w = img_bgr.shape[:2]
    if h < 200 or w < 200:
        return False, "resolucion_baja"

    aspect = w / float(h)
    if aspect < 0.35 or aspect > 2.85:
        return False, "aspect_ratio_extremo"

    # Filtro anti-catálogo (fondo blanco de estudio)
    borde = np.concatenate([
        img_bgr[0:3, :, :].reshape(-1, 3),
        img_bgr[-3:, :, :].reshape(-1, 3),
        img_bgr[:, 0:3, :].reshape(-1, 3),
        img_bgr[:, -3:, :].reshape(-1, 3)
    ])
    blanco_ratio = np.count_nonzero(np.mean(borde, axis=1) > 242) / float(len(borde))
    if blanco_ratio > 0.72:
        return False, "catalogo_blanco"

    # Filtro de contrastes y dibujos planos
    gris = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    desv = np.std(gris)
    if desv < 18:
        return False, "sin_contraste"

    return True, "OK"

print("✅ Módulo de limpieza y control de calidad cargado correctamente.")

## 5. Auditoría del Dataset: Estado y Visualización de Estadísticas
Verificación de conteos de imágenes válidas y descartadas por clase, junto a un gráfico de distribución.

In [ ]:
resumen = []
for cat in CATEGORIAS_10_CLASES:
    ruta_val = os.path.join(DIR_DATASET, cat)
    ruta_desc = os.path.join(DIR_DESCARTES, cat)
    num_val = len([f for f in os.listdir(ruta_val) if f.lower().endswith((".jpg", ".jpeg", ".png"))]) if os.path.exists(ruta_val) else 0
    num_desc = len([f for f in os.listdir(ruta_desc) if f.lower().endswith((".jpg", ".jpeg", ".png"))]) if os.path.exists(ruta_desc) else 0
    resumen.append({"categoria": cat, "limpias": num_val, "descartes": num_desc})

# Tabla de resultados
print(f"{'Categoría':<25} | {'Limpias':<10} | {'Descartes':<10} | {'Progreso'}")
print("-" * 65)
total_limpias = sum(r["limpias"] for r in resumen)
total_descartes = sum(r["descartes"] for r in resumen)
for r in resumen:
    pct = (r["limpias"] / META_POR_CLASE) * 100
    barra = "█" * int(pct // 10) + "░" * (10 - int(pct // 10))
    print(f"{r['categoria']:<25} | {r['limpias']:<10} | {r['descartes']:<10} | [{barra}] {pct:5.1f}%")
print("-" * 65)
print(f"{'TOTAL GENERAL':<25} | {total_limpias:<10} | {total_descartes:<10}")

# Gráfica de barras comparativa
cats = [r["categoria"] for r in resumen]
limpias = [r["limpias"] for r in resumen]
plt.figure(figsize=(12, 5))
plt.bar(cats, limpias, color="#2E7D32", edgecolor="black", alpha=0.85)
plt.axhline(1000, color="red", linestyle="--", label="Meta (1,000 por clase)")
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.ylabel("Imágenes Limpias Aprobadas")
plt.title("Distribución de Imágenes en el Dataset Suplementario")
plt.grid(axis="y", linestyle=":", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Muestra Visual de Muestras Aleatorias del Dataset

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for idx, cat in enumerate(CATEGORIAS_10_CLASES):
    ruta_cat = os.path.join(DIR_DATASET, cat)
    if os.path.exists(ruta_cat):
        fotos = [f for f in os.listdir(ruta_cat) if f.lower().endswith((".jpg", ".png", ".jpeg"))]
        if fotos:
            img_path = os.path.join(ruta_cat, fotos[0])
            img = Image.open(img_path).convert("RGB")
            axes[idx].imshow(img)
            axes[idx].set_title(cat, fontsize=10, fontweight="bold")
    axes[idx].axis("off")

plt.suptitle("Ejemplos Reales de las 10 Categorías Suplementarias", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Normalización y Recorte Inteligente con YOLO (180x180 px)

Tal como se requiere en el preprocesamiento avanzado:
1. **Sin distorsión ni relleno blanco (*Zero Letterbox/Padding*):** En lugar de redimensionar ingenuamente con barras laterales que agregan ruido al entrenamiento, se detecta el residuo con **YOLOv8**.
2. **Localización y encuadre del objeto:** Se obtiene el *bounding box* del residuo, se expande simétricamente a proporción 1:1 con un margen del 8% de contexto, y se recorta.
3. **Normalización exacta a 180x180:** El recorte se escala a 180x180 px mediante interpolación Lanczos.
4. **Mecanismo de Respaldo (*Fallback*):** En imágenes donde el residuo abarca todo el encuadre (texturas continuas de escombros o madera), se aplica *Center Crop* cuadrado sin franjas.

In [1]:
from ultralytics import YOLO
from PIL import Image
import cv2
import numpy as np

# Carga del modelo detector ligero YOLOv8
yolo_model = YOLO("yolov8n.pt")
TAMANO_ESTANDAR = (180, 180)

def recortar_y_normalizar_yolo(ruta_imagen, tamano=TAMANO_ESTANDAR, conf=0.15, padding_pct=0.08):
    img_bgr = cv2.imread(ruta_imagen)
    if img_bgr is None:
        return False
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H, W = img_rgb.shape[:2]
    
    # Inferencia con YOLO
    results = yolo_model(img_rgb, conf=conf, verbose=False)
    boxes = []
    for r in results:
        for b in r.boxes:
            boxes.append(b.xyxy[0].cpu().numpy())
            
    if boxes:
        # Envolvente de las cajas detectadas
        x1 = min(b[0] for b in boxes)
        y1 = min(b[1] for b in boxes)
        x2 = max(b[2] for b in boxes)
        y2 = max(b[3] for b in boxes)
        
        bw = x2 - x1
        bh = y2 - y1
        cx = (x1 + x2) / 2.0
        cy = (y1 + y2) / 2.0
        
        side = max(bw, bh) * (1.0 + 2.0 * padding_pct)
        side = min(side, max(H, W))
        
        cx1 = max(0, int(cx - side / 2.0))
        cy1 = max(0, int(cy - side / 2.0))
        cx2 = min(W, int(cx + side / 2.0))
        cy2 = min(H, int(cy + side / 2.0))
        
        crop = img_rgb[cy1:cy2, cx1:cx2]
        if crop.size == 0:
            crop = img_rgb
    else:
        # Fallback: Recorte central cuadrado (sin bordes blancos)
        side = min(H, W)
        cx1 = (W - side) // 2
        cy1 = (H - side) // 2
        crop = img_rgb[cy1:cy1 + side, cx1:cx1 + side]
        
    # Normalización final a 180x180 px RGB
    final_180 = cv2.resize(crop, tamano, interpolation=cv2.INTER_LANCZOS4)
    final_bgr = cv2.cvtColor(final_180, cv2.COLOR_RGB2BGR)
    cv2.imwrite(ruta_imagen, final_bgr, [cv2.IMWRITE_JPEG_QUALITY, 95])
    return True

print("✅ Pipeline de recorte con YOLO y normalización a 180x180 px configurado con éxito.")

✅ Pipeline de recorte con YOLO y normalización a 180x180 px configurado con éxito.


### 7.1 Ejecución Masiva del Recorte Inteligente con GPU (NVIDIA RTX 2050)
La siguiente celda recorre todas las imágenes de , detecta el residuo con YOLO, recorta el objeto en proporción 1:1 (con margen del 8%) y lo escala a **180x180 px** sin franjas blancas.

In [1]:
# Ejecutar la normalización masiva con GPU
!python normalizar_dataset_yolo.py


🚀 NORMALIZACIÓN INTELIGENTE CON YOLOv8 (180x180 px)
⚡ Dispositivo de aceleración: GPU (NVIDIA GeForce RTX 2050)
📁 Total de imágenes a normalizar: 6,789
Normalizando a 180x180:  83%|███████████▋  | 5663/6789 [02:02<00:22, 49.93it/s]libpng warning: sRGB: out of place
[ WARN:0@123.694] global loadsave.cpp:1121 imwrite_ An unsupported key(1) was specified and has been ignored.
Normalizando a 180x180:  86%|████████████  | 5837/6789 [02:05<00:16, 58.72it/s][ WARN:0@126.777] global loadsave.cpp:1121 imwrite_ An unsupported key(1) was specified and has been ignored.
[ WARN:0@126.857] global loadsave.cpp:1121 imwrite_ An unsupported key(1) was specified and has been ignored.
Normalizando a 180x180:  86%|████████████  | 5863/6789 [02:05<00:17, 54.08it/s][ WARN:0@127.247] global loadsave.cpp:1121 imwrite_ An unsupported key(1) was specified and has been ignored.
[ WARN:0@127.302] global loadsave.cpp:1121 imwrite_ An unsupported key(1) was specified and has been ignored.
Normalizando a 180x180:  8

### 7.2 Verificación de Dimensiones del Dataset Normalizado
Confirmamos que el 100% de las imágenes tengan ahora exactamente 180x180 px y 3 canales RGB.

In [2]:
from PIL import Image
import os

DIR = "dataset_suplementario"
tamanos_unicos = set()
modos_unicos = set()
total = 0

for cat in os.listdir(DIR):
    p_cat = os.path.join(DIR, cat)
    if os.path.isdir(p_cat):
        for f in os.listdir(p_cat):
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                total += 1
                with Image.open(os.path.join(p_cat, f)) as img:
                    tamanos_unicos.add(img.size)
                    modos_unicos.add(img.mode)

print(f"📊 Total de imágenes verificadas: {total:,}")
print(f"📐 Dimensiones encontradas en disco: {tamanos_unicos}")
print(f"🎨 Canales de color: {modos_unicos}")
if tamanos_unicos == {(180, 180)} and modos_unicos == {"RGB"}:
    print("🎉 ¡Éxito total! Todo el dataset está estandarizado en 180x180 px RGB listo para entrenar.")


📊 Total de imágenes verificadas: 6,789
📐 Dimensiones encontradas en disco: {(180, 180)}
🎨 Canales de color: {'RGB'}
🎉 ¡Éxito total! Todo el dataset está estandarizado en 180x180 px RGB listo para entrenar.
